In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
import plotly.graph_objects as go


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("princelv84/dogsvscats")

print("Path to dataset files:", path)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 545M/545M [01:10<00:00, 8.07MB/s] 

Extracting files...


Path to dataset files: /Users/admin/.cache/kagglehub/datasets/princelv84/dogsvscats/versions/1


In [27]:
model = Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
   
    
    Conv2D(128, (3,3), activation='relu'),
    
    
    Flatten(),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dense(1, activation='sigmoid')  
])

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 109, 109, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 54, 54, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 52, 52, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 26, 26, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_22 (Conv2D)              │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_23 (Conv2D)              │ (None, 10, 10, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_24 (Conv2D)              │ (None, 3, 3, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_25 (Conv2D)              │ (None, 1, 1, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │           513 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,497 (1.49 MB)

 Trainable params: 390,497 (1.49 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:

# Configuration
batch_size = 32
img_height = 224
img_width = 224

# Création du jeu d'entraînement
train_ds = tf.keras.utils.image_dataset_from_directory(
  'Datasets',
  validation_split=0.2, # 20% pour le test, 80% pour l'entraînement
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)

# Création du jeu de validation (test)
val_ds = tf.keras.utils.image_dataset_from_directory(
  'Datasets',
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size
)


Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Found 25000 files belonging to 2 classes.
Using 5000 files for validation.


In [ ]:
"""data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])"""

In [29]:
model.compile(
    optimizer='adam',               
    loss='binary_crossentropy',    
    metrics=['accuracy']           
)
##train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

In [30]:
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=10
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 189ms/step - accuracy: 0.4979 - loss: 0.6937 - val_accuracy: 0.4886 - val_loss: 0.6937
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 117s 187ms/step - accuracy: 0.5023 - loss: 0.6937 - val_accuracy: 0.4896 - val_loss: 0.6930
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 191ms/step - accuracy: 0.5062 - loss: 0.6932 - val_accuracy: 0.4886 - val_loss: 0.6938
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 124s 199ms/step - accuracy: 0.5006 - loss: 0.6932 - val_accuracy: 0.4886 - val_loss: 0.6937
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 128s 205ms/step - accuracy: 0.4986 - loss: 0.6932 - val_accuracy: 0.4886 - val_loss: 0.6936
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 191ms/step - accuracy: 0.5005 - loss: 0.6932 - val_accuracy: 0.4886 - val_loss: 0.6936
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 122s 196ms/step - accuracy: 0.5001 - loss: 0.6932 - val_accuracy: 0.4886 - val_loss: 0.6936
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 119s 190ms/step - accuracy: 0.5009 -

In [6]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [11]:
model.fit(train_ds,
  validation_data=val_ds,
  epochs=10)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 425s 680ms/step - accuracy: 0.9902 - loss: 0.0255 - val_accuracy: 0.9770 - val_loss: 0.0754
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 427s 683ms/step - accuracy: 0.9901 - loss: 0.0271 - val_accuracy: 0.9760 - val_loss: 0.0802
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 425s 680ms/step - accuracy: 0.9923 - loss: 0.0219 - val_accuracy: 0.9768 - val_loss: 0.0776
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 427s 684ms/step - accuracy: 0.9926 - loss: 0.0214 - val_accuracy: 0.9758 - val_loss: 0.0828
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 425s 681ms/step - accuracy: 0.9926 - loss: 0.0196 - val_accuracy: 0.9762 - val_loss: 0.1068
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 428s 685ms/step - accuracy: 0.9934 - loss: 0.0183 - val_accuracy: 0.9784 - val_loss: 0.0947
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 428s 686ms/step - accuracy: 0.9931 - loss: 0.0174 - val_accuracy: 0.9766 - val_loss: 0.0973
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 427s 683ms/step - accuracy: 0.9939 -